# ADS v2: Home acquisition events

Builds 28 event families: `HA1`, `HA2`, `HA2.F1`, `HA3`, `HA3.F1`, `HA0`, `HA0.F1`, `HA1.E2`, `HA1.E1`, `HA2.E1`, `HA1.F1`, `HA4a`, `HA4b`, `HA4.F1`, `HA5.F1`, `HA6.B1.1`, `HA6`, `HA6.B1.4`, `HA6.F1`, `HA7.B1.4`, `HA6.B1.5`, `HA6.B1.2`, `HA6.B1.3`, `HA6.B1.6`, `HA5.1`, `HA7`, `HA7.B1.1`, `HA6.B1.O1`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("13_Home_Acquisition", "started")


## HA1: Quote initiated ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hfq_quote                                                                   */
/*  Description : HFQ quotes created in the window, with the proposer email and mobile keys (stg.HA1_HF */
/*              : Q_Quotes).                                                                            */
/*              :                                                                                       */
/*              : Reads: ods_v2.Quote, ods_v2.QuoteHome                                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hfq_quote as
select  q.SourceSystemQuoteReference                                    as QuoteCodeReference,
        q.QuoteId,
        q.QuoteDateTime,
        h.EmailKey,
        h.MobileKey
from    ${ads.ods}.Quote q
left join ${ads.ods}.QuoteHome h    on h.SourceSystemId = 3 and h.SourceSystemQuoteReference = q.SourceSystemQuoteReference
where   q.SourceSystemId = 3
and     q.QuoteDateTime >= timestamp '${ads.start_time}' and q.QuoteDateTime < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha1  (HA1)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 38, 163. Every HFQ quote created in the window */
/*              : . Changes: the time is the quote's CreatedOn (ods_v2 reads dlk.hfq_quotedetails; 04 t */
/*              : ook the first load time of each retrieve version from the _ts of HFQ_QuoteDetails_Sna */
/*              : pshot_v2); source system 3 HFQ.                                                       */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: Quote, QuoteHome                               */
/*              : Reads: stg.adsv2_hfq_quote                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha1 as
select  xxhash64('HA1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA1'                                               as EventTypeId,
        'Quote initiated ONLINE' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  QuoteDateTime                                                   as EventDateTime,
                3                                                               as SourceSystemId,
                QuoteCodeReference                                              as SourceQuoteReference,
                QuoteId
        from    ${ads.stg}.adsv2_hfq_quote
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA2: Quote progressed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha2  (HA2)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 169. HFQ quotes with a price response in the wi */
/*              : ndow (quote progressed), at the first response.                                       */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: QuoteHomePrice                                 */
/*              : Reads: ods_v2.QuoteHomePrice                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha2 as
select  xxhash64('HA2', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA2'                                               as EventTypeId,
        'Quote progressed' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(DateCreated)                                                as EventDateTime,
                3                                                               as SourceSystemId,
                QuoteCodeReference                                              as SourceQuoteReference,
                max(QuoteId)                                                    as QuoteId
        from    ${ads.ods}.QuoteHomePrice
        where   DateCreated >= timestamp '${ads.start_time}' and DateCreated < timestamp '${ads.end_time}'
        group by QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA2.F1: We cannot see where customers drop out


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha2_f1  (HA2.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cells 14, 170. HFQ quotes that never requested a  */
/*              : price (no price response at all), at the quote time.                                  */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: Quote, QuoteHomePrice                          */
/*              : Reads: stg.adsv2_hfq_quote, ods_v2.QuoteHomePrice                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha2_f1 as
select  xxhash64('HA2.F1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA2.F1'                                            as EventTypeId,
        'We cannot see where customers drop out' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  q.QuoteDateTime                                                 as EventDateTime,
                3                                                               as SourceSystemId,
                q.QuoteCodeReference                                            as SourceQuoteReference,
                q.QuoteId
        from    ${ads.stg}.adsv2_hfq_quote q
        left anti join (select distinct QuoteCodeReference from ${ads.ods}.QuoteHomePrice) p
                on p.QuoteCodeReference = q.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA3: Home Acquisition - Customer gets a price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha3  (HA3)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 172. HFQ quotes with a premium returned in the  */
/*              : window, at the first one.                                                             */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: QuoteHomePrice                                 */
/*              : Reads: stg.adsv2_hfq_priced                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha3 as
select  xxhash64('HA3', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA3'                                               as EventTypeId,
        'Home Acquisition - Customer gets a price' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(DateCreated)                                                as EventDateTime,
                3                                                               as SourceSystemId,
                QuoteCodeReference                                              as SourceQuoteReference,
                max(QuoteId)                                                    as QuoteId
        from    ${ads.stg}.adsv2_hfq_priced
        where   DateCreated >= timestamp '${ads.start_time}' and DateCreated < timestamp '${ads.end_time}'
        group by QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA3.F1: Home Acquisition - No price returned, told to call us


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha3_f1  (HA3.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cell 173. HFQ quotes that requested a price (a pr */
/*              : ice response exists) and got no premium, at the first response. Change (agreed 2026-1 */
/*              : 0-15): 04 took every quote without a premium, so it also held HA2.F1's drop-offs, whi */
/*              : ch never asked for a price; the two no longer share quotes.                           */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: Quote, QuoteHomePrice                          */
/*              : Reads: stg.adsv2_hfq_quote, ods_v2.QuoteHomePrice, stg.adsv2_hfq_priced               */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha3_f1 as
select  xxhash64('HA3.F1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA3.F1'                                            as EventTypeId,
        'Home Acquisition - No price returned, told to call us' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(r.DateCreated)                                              as EventDateTime,
                3                                                               as SourceSystemId,
                q.QuoteCodeReference                                            as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_hfq_quote q
        join    ${ads.ods}.QuoteHomePrice r     on r.QuoteCodeReference = q.QuoteCodeReference
        left anti join (select distinct QuoteCodeReference from ${ads.stg}.adsv2_hfq_priced) p
                on p.QuoteCodeReference = q.QuoteCodeReference
        group by q.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA0: Straight into the CALL CENTRE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha0  (HA0)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 39, 167. Inbound conversations whose last inbo */
/*              : und queue session is INBOUND_SALES_HOME, from callers with no HFQ quote in the 30 day */
/*              : s before the call (D-E1.18: straight into the call centre). The caller is matched to  */
/*              : HFQ quotes on the mobile hash key. Change: source system 20 Genesys.                  */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession, Quote, QuoteHome                   */
/*              : Reads: ods_v2.CallSession, ods_v2.Quote, ods_v2.QuoteHome                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha0 as
select  xxhash64('HA0', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA0'                                               as EventTypeId,
        'Straight into the CALL CENTRE' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.ConversationStartTime)                                    as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference
        from    ${ads.ods}.CallSession c
        join    (
                select  ConversationId, max(SessionIndex) as SessionIndex
                from    ${ads.ods}.CallSession
                where   QueueName is not null and OriginatingDirection = 'inbound' and ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
                group by ConversationId
                ) l     on l.ConversationId = c.ConversationId and l.SessionIndex = c.SessionIndex
        left anti join (
                select  h.MobileKey, q.QuoteDateTime
                from    ${ads.ods}.Quote q
                join    ${ads.ods}.QuoteHome h  on h.SourceSystemId = 3 and h.SourceSystemQuoteReference = q.SourceSystemQuoteReference
                where   q.SourceSystemId = 3
                and     h.MobileKey is not null
                ) hq    on hq.MobileKey = c.AniPhoneKey
                       and hq.QuoteDateTime <= c.ConversationStartTime
                       and hq.QuoteDateTime >  c.ConversationStartTime - interval 30 days
        where   c.QueueName = 'INBOUND_SALES_HOME'
        and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'
        group by c.ConversationId
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA0.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha0_f1  (HA0.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cells 40, 168. Conversations on INBOUND_SALES_HOM */
/*              : E that waited more than 600 seconds or were abandoned (D-E1.25; wrap-up codes are not */
/*              :  used). Change: source system 20.                                                     */
/*              :                                                                                       */
/*              : Grain: Call   Product: Home   Source: CallSession                                     */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha0_f1 as
select  xxhash64('HA0.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA0.F1'                                            as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference
        from    ${ads.ods}.CallSession
        where   QueueName = 'INBOUND_SALES_HOME'
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        having  sum(AcdWaitDurationMs) > 600000 or sum(Abandoned) > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA1.E2: Outbound sales callback


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha1_e2  (HA1.E2)                                                        */
/*  Description : Acquisition / Chill Effort. Notebook 04 cell 165. Outbound sales callbacks to a mobil */
/*              : e, one event per customer number at its first call. Change: outbound conversations on */
/*              :  OUTBOUND_SALES_HOME (in use for outbound calls until 2024, ~28.8k conversations 2023 */
/*              : -24) or on INBOUND_SALES_HOME (outbound calls since then); agreed by Vijay 2026-10-19 */
/*              : ; 04 read OUTBOUND_SALES_HOME only. The mobile test is DnisIsMobile (04: the raw numb */
/*              : er like '08%').                                                                       */
/*              :                                                                                       */
/*              : Grain: Customer   Product: Home   Source: CallSession                                 */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha1_e2 as
select  xxhash64('HA1.E2', 'Customer',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA1.E2'                                            as EventTypeId,
        'Outbound sales callback' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Customer'                                          as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                DnisPhoneKey                                                    as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.ods}.CallSession
        where   QueueName in ('OUTBOUND_SALES_HOME', 'INBOUND_SALES_HOME')
        and     OriginatingDirection = 'outbound'
        and     DnisIsMobile = 1
        and     DnisPhoneKey is not null
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by DnisPhoneKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA1.E1: Service retrieve


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_home_retrieve_send                                                          */
/*  Description : Home retrieve / TYR campaign sends (email and SMS), tagged Service or Marketing by th */
/*              : e campaign name (D-E1.17).                                                            */
/*              :                                                                                       */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_home_retrieve_send as
select  EventDateTime, CampaignId, CampaignName, MessageType, EmailKey, QuoteGuidKey, PolicyCode,
        case when CampaignName like 'Home%TYR%- Service'
               or CampaignName in ('Home EQuote Retrieve Day 1', 'Home EQuote Retrieve Day 6') then 'Service'
             when CampaignName like 'Home%TYR%- Marketing' then 'Marketing' end as RetrieveType
from    ${ads.ods}.MessageEvent
where   InteractionType = 'sent'
and     MessageType in ('EMAIL', 'SMS')
and     CampaignName like 'Home%'
and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha1_e1  (HA1.E1)                                                        */
/*  Description : Acquisition / Chill Effort. Notebook 04 cell 164. Service retrieve sends (D-E1.17): t */
/*              : he '- Service' variants of the Home TYR campaigns and 'Home EQuote Retrieve Day 1 / D */
/*              : ay 6' (2022-23), email and SMS, one event per send. 04 read every 'Home % TYR %' emai */
/*              : l, mixing service and marketing, one row per email address for the whole window.      */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: MessageEvent                                   */
/*              : Reads: stg.adsv2_home_retrieve_send                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha1_e1 as
select  xxhash64('HA1.E1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA1.E1'                                            as EventTypeId,
        'Service retrieve' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  EventDateTime                                                   as EventDateTime,
                21                                                              as SourceSystemId,
                cast(CampaignId as string)                                      as EventSourceReference,
                QuoteGuidKey                                                    as SourceQuoteReference,
                PolicyCode                                                      as SourcePolicyReference,
                EmailKey                                                        as SourceCustomerReference,
                case when EmailKey is not null then 'EmailKey' end              as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_home_retrieve_send
        where   RetrieveType = 'Service'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA2.E1: Home Acquisition - TYR Campaign


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha2_e1  (HA2.E1)                                                        */
/*  Description : Acquisition / Chill EFfort. Notebook 04 cell 171. TYR marketing sends (D-E1.17): 'Hom */
/*              : e TYR 2.0 - Marketing', 'Home HFQ - TYR 2.1 - Marketing', 'Home TYR 0DTI - Quote Expi */
/*              : ry Journey - Marketing', email and SMS (04 missed the ~55k TYR 2.1 SMS), one event pe */
/*              : r send.                                                                               */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: MessageEvent                                   */
/*              : Reads: stg.adsv2_home_retrieve_send                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha2_e1 as
select  xxhash64('HA2.E1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA2.E1'                                            as EventTypeId,
        'Home Acquisition - TYR Campaign' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  EventDateTime                                                   as EventDateTime,
                21                                                              as SourceSystemId,
                cast(CampaignId as string)                                      as EventSourceReference,
                QuoteGuidKey                                                    as SourceQuoteReference,
                PolicyCode                                                      as SourcePolicyReference,
                EmailKey                                                        as SourceCustomerReference,
                case when EmailKey is not null then 'EmailKey' end              as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_home_retrieve_send
        where   RetrieveType = 'Marketing'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA1.F1: Home Acquisition - Outbound campaigns to quotes that never got a price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha1_f1  (HA1.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cell 166, corrected (agreed 2026-10-15): service  */
/*              : retrieve sends to quotes that never got a price. A send is matched to the recipient's */
/*              :  HFQ quotes on the email hash key, quotes created in the 30 days before the send (the */
/*              :  retrieve cadence); one event per send. 04 had it inverted (quotes whose proposer got */
/*              :  NO email).                                                                           */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: MessageEvent, Quote, QuoteHome, QuoteHomePrice */
/*              : Reads: stg.adsv2_home_retrieve_send, stg.adsv2_hfq_quote, stg.adsv2_hfq_priced        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha1_f1 as
select  xxhash64('HA1.F1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA1.F1'                                            as EventTypeId,
        'Home Acquisition - Outbound campaigns to quotes that never got a price' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  r.EventDateTime                                                 as EventDateTime,
                21                                                              as SourceSystemId,
                cast(r.CampaignId as string)                                    as EventSourceReference,
                max(q.QuoteCodeReference)                                       as SourceQuoteReference,
                r.EmailKey                                                      as SourceCustomerReference,
                'EmailKey'                                                      as SourceCustomerReferenceType
        from    ${ads.stg}.adsv2_home_retrieve_send r
        join    ${ads.stg}.adsv2_hfq_quote q    on q.EmailKey = r.EmailKey
                                               and q.QuoteDateTime <= r.EventDateTime
                                               and q.QuoteDateTime >  r.EventDateTime - interval 30 days
        left anti join (select distinct QuoteCodeReference from ${ads.stg}.adsv2_hfq_priced) p
                on p.QuoteCodeReference = q.QuoteCodeReference
        where   r.RetrieveType = 'Service'
        and     r.EmailKey is not null
        group by r.EventDateTime, r.CampaignId, r.EmailKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA4a: Customer buys ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha4a  (HA4a)                                                            */
/*  Description : Acquisition / Spine. Notebook 04 cells 33, 174, 177. The first successful HFQ payment */
/*              :  (latest response SUCCESS) on an HFQ quote of the window (agreed 2026-10-15: the firs */
/*              : t, not the last; HA5 merged into this step).                                          */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: Quote, QuoteHomePaymentResponse                */
/*              : Reads: stg.adsv2_hfq_quote, stg.adsv2_hfq_payment                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha4a as
select  xxhash64('HA4a', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA4a'                                              as EventTypeId,
        'Customer buys ONLINE' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(p.ActionTimeCreated)                                        as EventDateTime,
                3                                                               as SourceSystemId,
                q.QuoteCodeReference                                            as SourceQuoteReference,
                max(q.QuoteId)                                                  as QuoteId
        from    ${ads.stg}.adsv2_hfq_quote q
        join    ${ads.stg}.adsv2_hfq_payment p  on p.QuoteReference = q.QuoteCodeReference
        where   p.ActionResultCode = 'SUCCESS'
        and     p.ActionTimeCreated >= timestamp '${ads.start_time}' and p.ActionTimeCreated < timestamp '${ads.end_time}'
        group by q.QuoteCodeReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA4b: Agent quotes and converts on the CALL


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha4b  (HA4b)                                                            */
/*  Description : Acquisition / Spine. Notebook 04 cell 175, corrected (D-E1.15): new-business home sal */
/*              : es through a call-centre channel (a) IB, c) OB, a) Back Office) with no HEQReference. */
/*              :  Home sales never carry QuoteQueryGUID (0 of 87,387), so 04 counted every Home sale,  */
/*              : online included. HEQReference is the warehouse's Home eQuote / Relay-import reference */
/*              :  (DWH.usp_Load_fact_HEQ_Matching), not an HFQ quote code.                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale                                    */
/*              : Reads: stg.adsv2_policies_sold                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha4b as
select  xxhash64('HA4b', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA4b'                                              as EventTypeId,
        'Agent quotes and converts on the CALL' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(ReportingSaleDate) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_policies_sold
        where   ReportingSaleType = 'New Business'
        and     PolicyTypeGroup = 'Home'
        and     Channel in ('a) IB', 'c) OB', 'a) Back Office')
        and     HEQReference is null
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA4.F1: Web assist: starts online-  then calls to complete


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha4_f1  (HA4.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cell 176, corrected (D-E1.15): new-business home  */
/*              : sales through 'b) Web Assist' with an HEQReference (starts online, completes on a cal */
/*              : l). 04 tested QuoteQueryGuid, which Home sales never carry, so it returned no rows.   */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale                                    */
/*              : Reads: stg.adsv2_policies_sold                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha4_f1 as
select  xxhash64('HA4.F1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA4.F1'                                            as EventTypeId,
        'Web assist: starts online-  then calls to complete' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(min(ReportingSaleDate) as timestamp)                       as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference,
                max(HEQReference)                                               as SourceQuoteReference,
                max(PolicyId)                                                   as PolicyId
        from    ${ads.stg}.adsv2_policies_sold
        where   ReportingSaleType = 'New Business'
        and     PolicyTypeGroup = 'Home'
        and     Channel = 'b) Web Assist'
        and     HEQReference is not null
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA5.F1: Payment attempted with friction


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha5_f1  (HA5.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cell 178. HFQ payments whose latest response is D */
/*              : ECLINED in the window, first per quote.                                               */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Home   Source: QuoteHomePaymentResponse                       */
/*              : Reads: stg.adsv2_hfq_payment                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha5_f1 as
select  xxhash64('HA5.F1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA5.F1'                                            as EventTypeId,
        'Payment attempted with friction' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ActionTimeCreated)                                          as EventDateTime,
                3                                                               as SourceSystemId,
                QuoteReference                                                  as SourceQuoteReference,
                max(QuoteId)                                                    as QuoteId
        from    ${ads.stg}.adsv2_hfq_payment
        where   ActionResultCode = 'DECLINED'
        and     ActionTimeCreated >= timestamp '${ads.start_time}' and ActionTimeCreated < timestamp '${ads.end_time}'
        group by QuoteReference
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_home_chase                                                                  */
/*  Description : Home new-business chase snapshot rows with each row's own send date (DAY 1 / 7 / 14 / */
/*              :  20) and whether one of the six documents is outstanding.                             */
/*              :                                                                                       */
/*              : Reads: ods_v2.DocumentChaseHome                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_home_chase as
select  PolicyCode,
        Campaign,
        cast(case Campaign when 'DAY 1'  then try_cast(Day1SendDate as date)
                           when 'DAY 7'  then try_cast(Day7SendDate as date)
                           when 'DAY 14' then try_cast(Day14SendDate as date)
                           when 'DAY 20' then try_cast(Day20SendDate as date) end as timestamp) as SendDate,
        case when (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O') then 1 else 0 end                     as SixOutstanding
from    ${ads.ods}.DocumentChaseHome
where   PolicyType = 'New Business'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_1  (HA6.B1.1)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 185, corrected (D-E1.16): the DAY 1 chase send */
/*              :  (request issued) for a new-business home policy, at its send date. 04 stamped every  */
/*              : chase at the sale date.                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: stg.adsv2_home_chase                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_1 as
select  xxhash64('HA6.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.1'                                          as EventTypeId,
        'Document request issued' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_chase
        where   Campaign = 'DAY 1'
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6: Documents requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6  (HA6)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 180. Documents requested: the DAY 1 chase send  */
/*              : with one of six documents outstanding, at its send date (D-E1.16; 04 stamped it at th */
/*              : e sale date).                                                                         */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: stg.adsv2_home_chase                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6 as
select  xxhash64('HA6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6'                                               as EventTypeId,
        'Documents requested' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_chase
        where   Campaign = 'DAY 1'
        and     SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_4  (HA6.B1.4)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 187, corrected (D-E1.16): the first reminder i */
/*              : s the DAY 7 chase send with one of six documents still outstanding, at its send date. */
/*              :  04 read DAY 1 at the sale date (the same rows as HA6).                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: stg.adsv2_home_chase                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_4 as
select  xxhash64('HA6.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.4'                                          as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_chase
        where   Campaign = 'DAY 7'
        and     SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.F1: Number of chases before the customer submits


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_f1  (HA6.F1)                                                        */
/*  Description : Acquisition / Friction. Notebook 04 cell 181, corrected (D-E1.16): one event per chas */
/*              : e sent (DAY 1 / 7 / 14 / 20) while documents were still outstanding and before the fi */
/*              : rst accepted document, so the number of chases per policy is the count of these event */
/*              : s. DAY 20 counts here only. 04 kept one row per policy with more than one snapshot ro */
/*              : w.                                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome, DocumentWorkflowStatus     */
/*              : Reads: stg.adsv2_home_chase, ods_v2.DocumentWorkflowStatus                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_f1 as
select  xxhash64('HA6.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.F1'                                            as EventTypeId,
        'Number of chases before the customer submits' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  c.SendDate                                                      as EventDateTime,
                1                                                               as SourceSystemId,
                c.Campaign                                                      as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_chase c
        left join (
                select  PolicyCode, min(cast(EventTimestamp as timestamp)) as FirstAccepted
                from    ${ads.ods}.DocumentWorkflowStatus
                where   IsAccepted = true
                group by PolicyCode
                ) a     on a.PolicyCode = c.PolicyCode
        where   c.SixOutstanding = 1
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        and     (a.FirstAccepted is null or c.SendDate < a.FirstAccepted)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA7.B1.4: Non-arrival chase-  inbound


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha7_b1_4  (HA7.B1.4)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 194. The last chase send with one of six docum */
/*              : ents still outstanding (non-arrival chase), at its send date (D-E1.16; 04 stamped it  */
/*              : at the sale date).                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: DocumentChaseHome                             */
/*              : Reads: stg.adsv2_home_chase                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha7_b1_4 as
select  xxhash64('HA7.B1.4', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA7.B1.4'                                          as EventTypeId,
        'Non-arrival chase-  inbound' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                PolicyCode                                                      as SourcePolicyReference
        from    ${ads.stg}.adsv2_home_chase
        where   SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_5  (HA6.B1.5)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 188, corrected (D-E1.16): the escalation is th */
/*              : e Relay final chase, the first 'Chase Final Notice' email or letter actually sent (Em */
/*              : ailed Document, Document Transmitted or Document Posted Confirmation) after the sale; */
/*              :  it sits on the snapshot's DAY 14. 04 read the snapshot's non-DAY-1 rows at the sale  */
/*              : date.                                                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, PolicyDocumentEvent               */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_5 as
select  xxhash64('HA6.B1.5', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.5'                                          as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= cast(s.ReportingSaleDate as timestamp)
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDescription like '%Chase Final Notice%'
        and     (d.EventDescription like '%Emailed Document%' or d.EventDescription like '%Document Transmitted%'
                 or d.EventDescription like '%Document Posted Confirmation%')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_2  (HA6.B1.2)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 186. The first document upload after a new-bus */
/*              : iness home sale (agreed 2026-10-15; 04 took any upload in the window).                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, DocumentUpload                    */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentUpload                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_2 as
select  xxhash64('HA6.B1.2', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.2'                                          as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(u.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Home'
        and     cast(u.EventTimestamp as timestamp) >= cast(s.ReportingSaleDate as timestamp)
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_3  (HA6.B1.3)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 190. The first accepted document after a new-b */
/*              : usiness home sale (agreed 2026-10-15).                                                */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, DocumentWorkflowStatus            */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_3 as
select  xxhash64('HA6.B1.3', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.3'                                          as EventTypeId,
        'Documents validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Home'
        and     w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= cast(s.ReportingSaleDate as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_6  (HA6.B1.6)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 192. The same rule as HA6.B1.3 (as today; the  */
/*              : register's identical-steps row).                                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, DocumentWorkflowStatus            */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_6 as
select  xxhash64('HA6.B1.6', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.6'                                          as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(cast(w.EventTimestamp as timestamp))                        as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Home'
        and     w.IsAccepted = true
        and     cast(w.EventTimestamp as timestamp) >= cast(s.ReportingSaleDate as timestamp)
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA5.1: Terms of business sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha5_1  (HA5.1)                                                          */
/*  Description : Acquisition / Spine. Notebook 04 cell 179. Terms of business emailed or transmitted,  */
/*              : the first after the sale (04 took the latest in the window).                          */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, PolicyDocumentEvent               */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha5_1 as
select  xxhash64('HA5.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA5.1'                                             as EventTypeId,
        'Terms of business sent' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDateTime >= cast(s.ReportingSaleDate as timestamp)
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Emailed Document - Chill Terms of Business')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA7: Policy documents dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha7  (HA7)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 182. Policy documents dispatched (five terms-of */
/*              : -business descriptions), the first after the sale.                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, PolicyDocumentEvent               */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha7 as
select  xxhash64('HA7', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA7'                                               as EventTypeId,
        'Policy documents dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDateTime >= cast(s.ReportingSaleDate as timestamp)
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('New Business - Emailed Document - Terms Of Business',
                                       'New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Emailed Document - Chill Terms of Business',
                                       'New Business - Document Transmitted - Chill Terms of Business',
                                       'New Business - Emailed Document - 05E Terms Of Business')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA7.B1.1: Policy documents dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha7_b1_1  (HA7.B1.1)                                                    */
/*  Description : Acquisition / Branch. Notebook 04 cell 183. Any new-business home pack document (04's */
/*              :  list of document names) emailed or transmitted, the first after the sale. HA7.B1.2 a */
/*              : nd HA7.B1.3 (identical) are dropped as agreed. The three '... Cover Letter' names are */
/*              :  covered by '%Cover Letter', as in 04's own list.                                     */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, PolicyDocumentEvent               */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha7_b1_1 as
select  xxhash64('HA7.B1.1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA7.B1.1'                                          as EventTypeId,
        'Policy documents dispatched' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Home'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     d.EventDateTime >= cast(s.ReportingSaleDate as timestamp)
        and     s.ReportingSaleType = 'New Business'
        and     (d.EventDescription like 'New Business - Emailed Document - %'
                 or d.EventDescription like 'New Business - Document Transmitted - %')
        and     (   d.EventDescription like '%04 - Suitability Statement'
                 or d.EventDescription like '%05E Terms Of Business'
                 or d.EventDescription like '%205 SOF- Proposal_Placeholder'
                 or d.EventDescription like '%300 NB Home Email'
                 or d.EventDescription like '%Chill Terms of Business'
                 or d.EventDescription like '%Cover Letter'
                 or d.EventDescription like '%CV Checklist Link'
                 or d.EventDescription like '%CV Email Body'
                 or d.EventDescription like '%CV Suitability Statment.'
                 or d.EventDescription like '%Direct Debit Mandate'
                 or d.EventDescription like '%Document Checklist Email'
                 or d.EventDescription like '%Document Checklist SMS'
                 or d.EventDescription like '%Home LOI_Placeholder'
                 or d.EventDescription like '%Home Schedule_Placeholder'
                 or d.EventDescription like '%Home SOF / Proposal_Placeholder'
                 or d.EventDescription like '%Home Suitability Statement'
                 or d.EventDescription like '%Insurance Product Information Document'
                 or d.EventDescription like '%InsuranceProductInformationDocument'
                 or d.EventDescription like '%Insurer Doc 1_Placeholder'
                 or d.EventDescription like '%Insurer Doc 2_Placeholder'
                 or d.EventDescription like '%Insurer Doc 3_Placeholder'
                 or d.EventDescription like '%Interested Party'
                 or d.EventDescription like '%Interested Party - Allied Irish Bank plc(AIB Mortgage Bank)'
                 or d.EventDescription like '%Interested Party - Bank of Ireland (2 College Green)(Bank of Ireland)'
                 or d.EventDescription like '%Interested Party - Bank of Ireland Mortgage Bank U.C.(College Green)'
                 or d.EventDescription like '%Interested Party - Bank of Ireland Mortgages(Dublin - 2 College Green)'
                 or d.EventDescription like '%Interested Party - Bank of Ireland(College Green)'
                 or d.EventDescription like '%Interested Party - Bankinter SA(Maynooth)'
                 or d.EventDescription like '%Interested Party - EBS D.A.C(Artane)'
                 or d.EventDescription like '%Interested Party - Haven Mortgages Limited(Head Office)'
                 or d.EventDescription like '%Interested Party - Permanent TSB(Dublin - Head Office)'
                 or d.EventDescription like '%Interested Party - Permanent TSB(Head Office)'
                 or d.EventDescription like '%InterestedParty'
                 or d.EventDescription like '%Key Facts'
                 or d.EventDescription like '%Premium Breakdown'
                 or d.EventDescription like '%PremiumBreakdown'
                 or d.EventDescription like '%Quote Breakdown'
                 or d.EventDescription like '%Receipt'
                 or d.EventDescription like '%Receipt_Template'
                 or d.EventDescription like '%Schedule'
                 or d.EventDescription like '%Statement Of Fact'
                 or d.EventDescription like '%StatementOfFact'
                 or d.EventDescription like '%Terms Of Business')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## HA6.B1.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_ha6_b1_o1  (HA6.B1.O1)                                                  */
/*  Description : Acquisition / Outcome. Notebook 04 cell 191, corrected (as Motor A6.B1.O1): cancelled */
/*              :  for non-receipt = the policy's cancellation (stg.adsv2_hinl_cancellation, not custom */
/*              : er-requested), at its cancel date, on a new-business home policy with an insurer-led  */
/*              : cancellation email on or after its first final chase sent (HA6.B1.5). Before: the lat */
/*              : est such email on a policy with any final chase in the window, with no order between  */
/*              : them.                                                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Home   Source: PolicySale, PolicyCancellation, PolicyDocumen */
/*              : tEvent                                                                                */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_hinl_cancellation, stg.adsv2_relay_document */
/*              : _event                                                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_ha6_b1_o1 as
select  xxhash64('HA6.B1.O1', 'Policy',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'HA6.B1.O1'                                         as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Home'                                              as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(x.CancelDateTime)                                           as EventDateTime,
                1                                                               as SourceSystemId,
                s.PolicyCode                                                    as SourcePolicyReference,
                max(s.PolicyId)                                                 as PolicyId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_hinl_cancellation x on x.PolicyCode = s.PolicyCode
        join    (
                select  e.PolicyCode, min(e.EventDateTime) as FinalChase
                from    ${ads.stg}.adsv2_relay_document_event e
                where   e.PolicyTypeGroup = 'Home'
                and     e.EventDescription like '%Chase Final Notice%'
                and     (e.EventDescription like '% - Emailed Document - %' or e.EventDescription like '% - Document Transmitted - %'
                         or e.EventDescription like '% - Document Posted Confirmation - %')
                group by e.PolicyCode
                ) c     on c.PolicyCode = s.PolicyCode
        left semi join ${ads.stg}.adsv2_relay_document_event d
                on  d.PolicyCode = s.PolicyCode
                and d.PolicyTypeGroup = 'Home'
                and d.EventDescription in ('Insurer Led Cancelation - Emailed Document - Reg canx email template',
                                           'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template')
                and d.EventDateTime >= c.FinalChase
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Home'
        and     not (coalesce(x.Reason, '') like 'Customer%' or coalesce(x.Reason, '') like 'Client%')
        and     x.CancelDateTime >= cast(s.ReportingSaleDate as timestamp)
        and     x.CancelDateTime >= timestamp '${ads.start_time}' and x.CancelDateTime < timestamp '${ads.end_time}'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_ha1", "adsv2_evt_ha2", "adsv2_evt_ha2_f1", "adsv2_evt_ha3", "adsv2_evt_ha3_f1", "adsv2_evt_ha0", "adsv2_evt_ha0_f1", "adsv2_evt_ha1_e2", "adsv2_evt_ha1_e1", "adsv2_evt_ha2_e1", "adsv2_evt_ha1_f1", "adsv2_evt_ha4a", "adsv2_evt_ha4b", "adsv2_evt_ha4_f1", "adsv2_evt_ha5_f1", "adsv2_evt_ha6_b1_1", "adsv2_evt_ha6", "adsv2_evt_ha6_b1_4", "adsv2_evt_ha6_f1", "adsv2_evt_ha7_b1_4", "adsv2_evt_ha6_b1_5", "adsv2_evt_ha6_b1_2", "adsv2_evt_ha6_b1_3", "adsv2_evt_ha6_b1_6", "adsv2_evt_ha5_1", "adsv2_evt_ha7", "adsv2_evt_ha7_b1_1", "adsv2_evt_ha6_b1_o1"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("13_Home_Acquisition", "succeeded")
